# Using RNN to build an IMBD review classifier

#### Basic flow of steps
1. Feature Engineering
2. Training Simple RNN with Embeddings Layer
3. Predictions using trained RNN
4. Deployment using Streamlit

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout

##### Loading the data set

In [2]:
max_features = 10000 #vocabulary size
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print(f'Training data shape: {x_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {x_test.shape}, Testing labels shape: {y_test.shape}')

Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [3]:
from tensorflow.keras.preprocessing import sequence

max_len = 500 #Maximum lenght for characters

x_train = sequence.pad_sequences(x_train, maxlen = max_len)
x_test = sequence.pad_sequences(x_test, maxlen = max_len)

### Training our model

In [4]:
model = Sequential()
model.add(Embedding(max_features, 128, input_length=max_len))
model.add(SimpleRNN(128, activation='tanh'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.build(input_shape=(None, max_len))
model.summary()

c:\ML bootcamp\Simple RNN\venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

Creating an instance of Earlystopping call back

In [5]:
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [7]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [8]:
# Training the model

histroy = model.fit(
    x_train, y_train, epochs=10, batch_size = 32,
    validation_split = 0.2,
    callbacks = [early_stopping]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 87ms/step - accuracy: 0.5405 - loss: 0.7179 - val_accuracy: 0.6210 - val_loss: 0.6492
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 54s 86ms/step - accuracy: 0.6374 - loss: 0.6350 - val_accuracy: 0.6860 - val_loss: 0.6009
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.7520 - loss: 0.5206 - val_accuracy: 0.7200 - val_loss: 0.5758
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.7804 - loss: 0.4777 - val_accuracy: 0.7224 - val_loss: 0.5921
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 89ms/step - accuracy: 0.8284 - loss: 0.4087 - val_accuracy: 0.7260 - val_loss: 0.5810
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 89ms/step - accuracy: 0.8190 - loss: 0.4127 - val_accuracy: 0.7248 - val_loss: 0.5992
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.8533 - loss: 0.3591 - val_accuracy: 0.7352 - val_loss: 0.6341
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.7236 - loss: 0.5196 - 

In [9]:
model.save('Simple_RNN_IMDB.keras')